In [4]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath(".."))
from braindecode.models import EEGNet
from model import EEGVAE
from eeg_preprocessing.loaders import build_loaders
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

DATASETS = ["BCI2A"]
RUN_RAW = True
RUN_FEATURES = True
N_EPOCHS = 50
LR = 1e-3
BACKBONE_CKPT = "/home/infres/yrothlin-24/EEG_Foundation_Model/outputs/pretraining/without_augmentation/pretraining_best.pt"

USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = {
    "SLEEPEDFX": [0.54, 1.88, 1.09, 2.86, 1.83], 
}

datasets = {
    "PHYSIONETMI": {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/PHYSIONETMI/physionetmi.lmdb", "n_classes": 4},
    "SLEEPEDFX":   {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SLEEPEDF/sleepedf.lmdb", "n_classes": 5},
    "SEEDV":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SEEDV/seedv.lmdb", "n_classes": 5},
    "BCI2A":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/BCI2A/bci2a.lmdb", "n_classes": 4},
    "TUAB":        {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/TUAB/tuab.lmdb", "n_classes": 2},
    "FACED":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/FACED/faced.lmdb", "n_classes": 9},
    "KARAONE":     {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/KARAONE/karaone.lmdb", "n_classes": 11},
    "BCI2020":     {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/BCI2020/bci2020.lmdb", "n_classes": 5},
    "CHISCO":      {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/CHISCO/chisco.lmdb", "n_classes": 39},
    "SIENA":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SIENA/siena.lmdb", "n_classes": 2},
    "SHUMI":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SHUMI/shumi.lmdb", "n_classes": 2},
}

data_config = {
    "batch_size": 64,
    "split_ratio": [0.8, 0.1, 0.1],
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": False,
    "seed": 100,
    "channel_mode": "mapped",
    "return_ch_names": False,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(100)
print("device:", device, "| datasets:", DATASETS)


device: cuda | datasets: ['BCI2A']


In [5]:
def make_loaders(name, channel_mode=None):
    return build_loaders(
        lmdb_path=datasets[name]["dataset_path"],
        split_ratio=data_config["split_ratio"],
        batch_size=data_config["batch_size"],
        seed=data_config["seed"],
        num_workers=data_config["num_workers"],
        pin_memory=data_config["pin_memory"],
        persistent_workers=data_config["persistent_workers"],
        channel_mode=channel_mode or data_config["channel_mode"],
        return_ch_names=data_config["return_ch_names"],
    )


def class_weights_for(name, train_loader, n_classes):
    """Return a CrossEntropy weight tensor on `device`, or None for an unweighted loss."""
    if not USE_CLASS_WEIGHTS:
        return None
    spec = CLASS_WEIGHTS.get(name)
    if spec is None:
        return None
    if spec == "auto":
        counts = torch.zeros(n_classes)
        for _, y in train_loader:
            counts += torch.bincount(y.long().flatten(), minlength=n_classes)
        counts = counts.clamp_min(1.0)
        weights = counts.sum() / (n_classes * counts)
    else:
        weights = torch.tensor(spec, dtype=torch.float32)
    return weights.to(device)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        y_pred.append(model(x.to(device).float()).argmax(1).cpu())
        y_true.append(y)
    return torch.cat(y_true), torch.cat(y_pred)


def train_eegnet(train_loader, val_loader, n_chans, n_times, n_classes, class_weights=None, **eegnet_kwargs):
    model = EEGNet(n_chans=n_chans, n_outputs=n_classes, n_times=n_times, **eegnet_kwargs).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=N_EPOCHS, steps_per_epoch=len(train_loader))
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_bacc, best_state = 0.0, None
    for epoch in range(N_EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device).float(), y.to(device).long()
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            scheduler.step()
        yt, yp = evaluate(model, val_loader)
        val_bacc = balanced_accuracy_score(yt, yp)
        if val_bacc > best_bacc:
            best_bacc = val_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


def scores(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro"),
    }


In [6]:
def load_backbone(ckpt_path):
    config = {
        "in_channels": 19, "out_channels": 19, "adaptor_layers": 4,
        "z_channels": 64, "ch": 64, "ch_mult": [1, 2, 4, 8], "n_blocks": 2,
        "dropout": 0.0, "mode": "vq", "num_embeddings": 256, "n_quantizers": 8,
        "commitment_cost": 0.25, "decay": 0.99,
    }
    sd = torch.load(ckpt_path, map_location="cpu")["model"]
    q = [k for k in sd
         if k.startswith("autoencoder.posterior.quantize.quantizers.") and k.endswith(".embedding")]
    config["z_channels"] = sd[q[0]].shape[1]
    config["n_quantizers"] = len({int(k.split(".quantizers.")[1].split(".")[0]) for k in q})
    model = EEGVAE(**config).to(device).eval()
    model.load_state_dict(sd)
    for p in model.parameters():
        p.requires_grad = False
    return model


@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for x, y in loader:
        feats.append(backbone.encode(x.to(device).float()).mode().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)


def feature_loaders(train_loader, val_loader, test_loader):
    B = data_config["batch_size"]
    ztr, ytr = extract_features(train_loader)
    zva, yva = extract_features(val_loader)
    zte, yte = extract_features(test_loader)
    loaders = (
        DataLoader(TensorDataset(ztr, ytr), batch_size=B, shuffle=True),
        DataLoader(TensorDataset(zva, yva), batch_size=B),
        DataLoader(TensorDataset(zte, yte), batch_size=B),
    )
    return loaders, ztr.shape[1], ztr.shape[2]


FEAT_KW = dict(kernel_length=16, depthwise_kernel_length=8,
               pool1_kernel_size=2, pool2_kernel_size=4)

backbone = load_backbone(BACKBONE_CKPT) if RUN_FEATURES else None


In [8]:

raw_results = []
for name in DATASETS:
    n_classes = datasets[name]["n_classes"]
    train_loader, val_loader, test_loader = make_loaders(name)
    xb, _ = next(iter(train_loader))
    n_chans, n_times = xb.shape[1], xb.shape[2]
    cw = class_weights_for(name, train_loader, n_classes)
    print(f"\n### {name} | raw | n_chans={n_chans} n_times={n_times} n_classes={n_classes}"
          + (" | weighted CE" if cw is not None else ""))

    model = train_eegnet(train_loader, val_loader, n_chans, n_times, n_classes, class_weights=cw)
    s = scores(*evaluate(model, test_loader))
    raw_results.append({"dataset": name, "mode": "raw", **s})
    print(f"  raw      -> acc={s['acc']:.3f} bacc={s['bacc']:.3f} f1={s['f1']:.3f}")


[LOADER] Total subjects: 9
[LOADER] Split source: ratio
[LOADER] Train subjects: 7 | ['A05', 'A01', 'A06', 'A09', 'A07', 'A02', 'A04']
[LOADER] Validation subjects: 1 | ['A08']
[LOADER] Test subjects: 1 | ['A03']

### BCI2A | raw | n_chans=19 n_times=800 n_classes=4
  raw      -> acc=0.716 bacc=0.717 f1=0.717


In [ ]:
feat_results = []
for name in DATASETS:
    n_classes = datasets[name]["n_classes"]
    train_loader, val_loader, test_loader = make_loaders(name, channel_mode="mapped")

    (f_train, f_val, f_test), z_chans, z_times = feature_loaders(train_loader, val_loader, test_loader)
    cw = class_weights_for(name, f_train, n_classes)
    print(f"\n### {name} | features EEGVAE | z_chans={z_chans} z_times={z_times} n_classes={n_classes}"
          + (" | weighted CE" if cw is not None else ""))

    model = train_eegnet(f_train, f_val, z_chans, z_times, n_classes, class_weights=cw, **FEAT_KW)
    s = scores(*evaluate(model, f_test))
    feat_results.append({"dataset": name, "mode": "features", **s})
    print(f"  features -> acc={s['acc']:.3f} bacc={s['bacc']:.3f} f1={s['f1']:.3f}")


[LOADER] Total subjects: 39
[LOADER] Split source: ratio
[LOADER] Train subjects: 31 | ['SC466', 'SC402', 'SC425', 'SC453', 'SC443', 'SC459', 'SC436', 'SC444', 'SC467', 'SC457', 'SC461', 'SC464', 'SC430', 'SC440', 'SC470', 'SC415', 'SC416', 'SC463', 'SC477', 'SC462', 'SC423', 'SC418', 'SC448', 'SC449', 'SC405', 'SC420', 'SC433', 'SC452', 'SC407', 'SC408', 'SC441']
[LOADER] Validation subjects: 4 | ['SC419', 'SC458', 'SC451', 'SC454']
[LOADER] Test subjects: 4 | ['SC424', 'SC471', 'SC460', 'SC422']

### SLEEPEDFX | features EEGVAE | z_chans=256 z_times=750 n_classes=5 | weighted CE


KeyboardInterrupt: 

## Résultats — EEGNet brut vs features EEGVAE

In [ ]:
results = pd.DataFrame(globals().get("raw_results", []) + globals().get("feat_results", []))
display(results.round(3))

pivot = results.pivot(index="dataset", columns="mode", values="bacc")
ax = pivot.plot(kind="bar", figsize=(1.8 * len(pivot) + 2, 4))
ax.set_ylabel("balanced accuracy (test)")
ax.set_title("EEGNet — brut vs features EEGVAE")
ax.set_ylim(0, 1)
ax.legend(title="")
plt.tight_layout()
plt.show()
